In [25]:
import jax.numpy as jnp

from chex import Array
from flax import linen as nn

import optax
import numpy as np

from tqdm import tqdm

from bandit import *

from reinforced_lib import RLib
from reinforced_lib.agents.deep import PPODiscrete

In [26]:
# ============================================================
# PPO NETWORK
# ============================================================
class PPONetwork(nn.Module):

    n_actions: int = 5

    @nn.compact
    def __call__(self, x):

        x = nn.Dense(64)(x)
        x = nn.relu(x)

        x = nn.Dense(64)(x)
        x = nn.relu(x)


        # Actor
        logits = nn.Dense(self.n_actions)(x)


        # Critic
        value = nn.Dense(1)(x)

        # IMPORTANT
        value = jnp.squeeze(value, axis=-1)

        return logits, value

In [27]:
# ============================================================
# CREATE PPO AGENT
# ============================================================

rl = RLib(
    agent_type=PPODiscrete,

    agent_params={

        # ----------------------------------------------------
        # Network
        # ----------------------------------------------------

        "network": PPONetwork(n_actions=5),

        "obs_space_shape": (2,),
        "act_space_size": 5,


        # ----------------------------------------------------
        # Optimizer
        # ----------------------------------------------------

        "optimizer": optax.adam(1e-3),


        # ====================================================
        # CONTEXTUAL BANDIT CONFIGURATION
        # ====================================================
        #
        # Our next state is independent of our action.
        #
        # So:
        #
        #     s_t --a_t--> reward
        #
        # but:
        #
        #     a_t does NOT influence s_(t+1)
        #
        # Therefore we initially use:
        #
        #     gamma = 0
        #
        # This makes the learning target focus on the
        # immediate reward.
        # ====================================================

        "discount": 0.0,

        # With gamma = 0, GAE effectively reduces to the
        # immediate TD-style advantage:
        #
        #     A = r - V(s)
        #
        # lambda_gae therefore does not materially affect
        # future-credit propagation in this experiment.
        "lambda_gae": 0.9,


        # ----------------------------------------------------
        # Advantage normalization
        # ----------------------------------------------------

        "normalize_advantage": True,


        # ----------------------------------------------------
        # PPO clipping
        # ----------------------------------------------------

        # Restricts how aggressively the policy changes.
        #
        # IMPORTANT:
        #
        # This is NOT epsilon-greedy exploration.
        #
        # PPO exploration comes from stochastic action
        # sampling + entropy regularization.
        "clip_coef": 0.2,


        # ----------------------------------------------------
        # Value function clipping
        # ----------------------------------------------------

        "clip_value": True,


        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        "clip_grad": 0.5,


        # ----------------------------------------------------
        # Entropy bonus
        # ----------------------------------------------------
        #
        # Encourages the policy to remain exploratory.
        #
        # Larger value:
        #     more stochasticity / exploration
        #
        # Smaller value:
        #     policy becomes deterministic faster
        # ----------------------------------------------------

        "entropy_coef": 0.01,


        # ----------------------------------------------------
        # Critic loss coefficient
        # ----------------------------------------------------

        "value_coef": 0.5,


        # ====================================================
        # ROLLOUT CONFIGURATION
        # ====================================================
        #
        # We deliberately keep this SMALL.
        #
        # We want to SEE PPO updating.
        #
        # Flow:
        #
        #     collect 16 interactions
        #             ↓
        #       compute GAE
        #             ↓
        #       compute returns
        #             ↓
        #       PPO updates
        #
        # Then repeat.
        # ====================================================

        "rollout_length": 16,


        # One environment only.
        "num_envs": 1,


        # PPO splits the rollout into mini-batches.
        "batch_size": 8,


        # Number of passes over the rollout data.
        "num_epochs": 4,
    },

    # We are NOT using Gymnasium.
    no_ext_mode=True
)



In [38]:
# ============================================================
# INITIAL STATE
# ============================================================

# PPO expects a batch dimension:
#
#     (num_envs, state_dimension)
#
# We have one environment.
#
# Therefore:
#
#     (1, 2)
# ============================================================

state = np.zeros((1, 2), dtype=np.float32)


# ============================================================
# INITIAL ACTION
# ============================================================

action = rl.sample(
    is_training=False,

    sample_observations={
        "env_states": state,
    },
)

action = np.asarray(action)


# ============================================================
# TRAINING LOOP
# ============================================================

n_steps = 10_000

count = 0


for step in tqdm(range(n_steps)):

    # ========================================================
    # STEP 1: ENVIRONMENT INTERACTION
    # ========================================================

    # Extract the single environment from the PPO batch.
    env_state = state[0]

    # Extract the single action.
    env_action = int(action[0])


    reward = get_reward(
        rng,
        env_state,
        env_action
    )


    # ========================================================
    # STEP 2: GENERATE NEXT CONTEXT
    # ========================================================

    env_next_state = sample_state(rng).astype(np.float32)


    # Add batch dimension for PPO.
    #
    # (2,) -> (1, 2)
    #
    next_state = env_next_state[None, :]


    # ========================================================
    # STEP 3: PREPARE BATCHED REWARD
    # ========================================================

    rewards = np.asarray(
        [reward],
        dtype=np.float32
    )


    terminals = np.asarray(
        [False]
    )


    # ========================================================
    # STEP 4: UPDATE PPO + SAMPLE NEXT ACTION
    # ========================================================

    next_action = rl.sample(

        update_observations={

            "env_states": next_state,

            # Shape:
            #
            # (1,)
            #
            "actions": action,

            # Shape:
            #
            # (1,)
            #
            "rewards": rewards,

            # Shape:
            #
            # (1,)
            #
            "terminals": terminals,
        },


        sample_observations={
            "env_states": next_state,
        }
    )


    next_action = np.asarray(next_action)


    # ========================================================
    # STEP 5: EVALUATION
    # ========================================================

    env_next_action = int(next_action[0])

    if env_next_action == optimal_action(env_next_state):
        count += 1


    # ========================================================
    # STEP 6: MOVE FORWARD
    # ========================================================

    state = next_state
    action = next_action

# ============================================================
# FINAL RESULT
# ============================================================

print("\nFinal optimal-action accuracy:")
print(count / n_steps)

100%|██████████| 10000/10000 [00:04<00:00, 2164.86it/s]


Final optimal-action accuracy:
0.8802
